In [1]:
import pandas as pd
import numpy as np

In [2]:
df_total = pd.read_excel("БАЗА Кварк _ Основная _ 02.05_2026.xlsx")

In [71]:

df_total['Пол'].map({1: 1, 2:0})

0       1.0
1       1.0
2       1.0
3       1.0
4       1.0
       ... 
4033    NaN
4034    0.0
4035    1.0
4036    0.0
4037    NaN
Name: Пол, Length: 4038, dtype: float64

In [72]:
df_total[[col for col in df_total.columns if ('ФВ' in col)]]

,"ФВ, %",ФВ ниже нормы,ФВ ниже 40%
0,59.0,0.0,0.0
1,59.0,0.0,0.0
2,39.0,1.0,1.0
3,39.0,1.0,1.0
4,58.0,0.0,0.0
...,...,...,...
4033,62.0,0.0,0.0
4034,72.0,0.0,0.0
4035,59.0,0.0,0.0
4036,60.0,0.0,0.0


In [73]:
df_total['ФВ ниже 55%'] = (df_total['ФВ, %'] < 55).astype(int)

df_total['ФВ ниже 30%'] = (df_total['ФВ, %'] < 30).astype(int)

In [74]:
df_total[['Пол', 'ФВ, %',
 'ДД 1 степени',
 'ДД2-3']]

df_total['ФВ, %'] = df_total['ФВ, %'].fillna(df_total['ФВ, %'])


df_total['DyastDisf_outcome'] = np.where(df_total['ДД2-3'] == 1, 1, 0)

df_total['DyastDisf_outcome']

0       0
1       0
2       0
3       0
4       1
       ..
4033    0
4034    0
4035    0
4036    0
4037    0
Name: DyastDisf_outcome, Length: 4038, dtype: int64

In [75]:
#rows_mask = df_total[['ДД 1 степени',
rows_mask = df_total[[ 'ДД2-3']].isna().all(axis=1)
cols_to_select = ['DyastDisf_outcome', 'Возраст, годы', 'Пол', 'Рост, см', 'Вес, кг', 'Курение', 'Гипертоническая болезнь', 'Сахарный диабет'] + df_total.loc[:, 'RR':'SDNN'].columns.tolist()

min_filled = int(len(cols_to_select) * 0.8)
df_dyastole = df_total.loc[~rows_mask, cols_to_select].dropna(thresh=min_filled)
# 'DyastDisf_outcome' - бинарный исход

In [76]:
cols_for_key = [
    'Возраст, годы',
    'Рост, см',
    'Вес, кг',
    'СИС АД при поступлении, мм.рт.ст',
    'ДИА АД при поступлении, мм.рт.ст.',
    'ФВ, %'
]

df_total['cols_for_key'] = df_total[cols_for_key].sum(axis = 1)
df_total['cols_for_key'] = df_total['cols_for_key'].astype(int)
df_total['cols_for_key'] = df_total['cols_for_key'].fillna(0).astype(int)

In [10]:
# !pip install xgboost
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix, 
                             classification_report, balanced_accuracy_score)
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import VotingClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

In [122]:
#df_dyastole = df_dyastole[df_dyastole['RR'] != 0]

from sklearn.model_selection import train_test_split
X = df_dyastole.loc[:, 'Возраст, годы':'SDNN']
y = df_dyastole['DyastDisf_outcome'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True, stratify=y
)
custom_weights = {0: 1, 1: 2}

model_lr = LogisticRegressionCV(penalty = 'elasticnet', solver = 'saga', max_iter = 8000, Cs=[0.1])
model_tr = ExtraTreesClassifier(n_estimators = 800, class_weight = custom_weights, random_state = 42)

ensemble = VotingClassifier(
    estimators=[('lr', model_lr), ('tr', model_tr)],
    voting = 'soft'
)

cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

Pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('clf', ensemble)
])


param_grid = {
    'clf__tr__max_depth': [10],
    'clf__lr__l1_ratios': [[0.1, 0.5, 0.9]],
}
    

grid_search = HalvingGridSearchCV(Pipe, param_grid, cv = cv_strategy, scoring = 'roc_auc', resource = 'n_samples', factor = 3, n_jobs = -1, random_state =42)
grid_search.fit(X_train,y_train)
best_xgb = grid_search.best_estimator_


calibrated_model = CalibratedClassifierCV(best_xgb, method='isotonic', cv=5, n_jobs = -1)
calibrated_model.fit(X_train, y_train)
    

train_probs = calibrated_model.predict_proba(X_train)[:, 1]
fpr, tpr, thresholds = roc_curve(y_train, train_probs)

# --- ИЗМЕНЕНИЕ: Внедрение веса для максимизации Чувствительности (Se) ---
# Настройка: w = 4 означает, что пропустить больного в 4 раза хуже, чем ошибиться на здоровом.
# Вы можете менять это число (например, поставить 3 или 5), чтобы регулировать баланс Se и Sp.
w = 3

# Рассчитываем взвешенный индекс Юдена вместо стандартного (tpr - fpr)
weighted_youden_index = (w * tpr) - fpr
optimal_idx = np.argmax(weighted_youden_index)
youden_threshold = thresholds[optimal_idx]
# ------------------------------------------------------------------------

# 2. Применяем новый смещенный порог на тестовой выборке
test_probs = calibrated_model.predict_proba(X_test)[:, 1]
y_pred_custom = (test_probs >= youden_threshold).astype(int)
  

/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/an

In [123]:
 # Расчет метрик
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
metrics = {
    'AUC': roc_auc_score(y_test, test_probs),
    'Se (Sens)': tp / (tp + fn),
    'Sp (Spec)': tn / (tn + fp),
    'PPV': tp / (tp + fp),
    'NPV': tn / (tn + fn),
    'Balanced Acc': balanced_accuracy_score(y_test, y_pred_custom),
    'Youden Threshold': youden_threshold
}
    
    



print(metrics)

{'AUC': 0.8369047619047619, 'Se (Sens)': np.float64(0.7333333333333333), 'Sp (Spec)': np.float64(0.7523809523809524), 'PPV': np.float64(0.24087591240875914), 'NPV': np.float64(0.9634146341463414), 'Balanced Acc': 0.7428571428571429, 'Youden Threshold': np.float64(0.10090979153506763)}


In [267]:
print(grid_search.best_params_)

{'clf__lr__l1_ratios': [0.1, 0.5, 0.9], 'clf__tr__max_depth': 10}


In [268]:
y.value_counts()


DyastDisf_outcome
0    2102
1     223
Name: count, dtype: int64

In [269]:
train_probs = calibrated_model.predict_proba(X_train)[:, 1]
y_train_custom = (train_probs >= youden_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_train, y_train_custom).ravel()
metrics = {
    'AUC': roc_auc_score(y_train, train_probs),
    'Se (Sens)': tp / (tp + fn),
    'Sp (Spec)': tn / (tn + fp),
    'PPV': tp / (tp + fp),
    'NPV': tn / (tn + fn),
    'Balanced Acc': balanced_accuracy_score(y_train, y_train_custom),
    'Youden Threshold': youden_threshold
}
    
    



print(metrics)

{'AUC': 0.9873495303878476, 'Se (Sens)': np.float64(1.0), 'Sp (Spec)': np.float64(0.8686087990487514), 'PPV': np.float64(0.44611528822055135), 'NPV': np.float64(1.0), 'Balanced Acc': 0.9343043995243757, 'Youden Threshold': np.float64(0.11122606581469255)}


In [77]:
df_total['SystDisf_outcome_AV'] = np.where(
    ((df_total['Пол'] == 1) & (df_total['ФВ, %'] < 52)) |
    ((df_total['Пол'] == 0) & (df_total['ФВ, %'] < 54)),
    1,
    0
)

# Целевая переменная 2: ФВ ниже 40% (тяжелая)
df_total['SystDisf_outcome_AW'] = df_total['ФВ ниже 40%'].fillna(0).astype(int)

rows_mask = df_total[['ФВ ниже 40%','ФВ ниже 30%', 'ФВ, %']].isna().all(axis=1)
cols_to_select = ['SystDisf_outcome_AV', 'SystDisf_outcome_AW', 'Возраст, годы', 'Пол', 'Рост, см', 'Вес, кг', 'Курение', 'Гипертоническая болезнь', 'Сахарный диабет', 'cols_for_key' ] + df_total.loc[:, 'RR':'SDNN'].columns.tolist()
min_filled = int(len(cols_to_select) * 0.8)
df_systole = df_total.loc[~rows_mask, cols_to_select].dropna(thresh=min_filled)

df_systole
# 'SystDisf_outcome' - бинарный исход

,SystDisf_outcome_AV,SystDisf_outcome_AW,"Возраст, годы",Пол,"Рост, см","Вес, кг",Курение,Гипертоническая болезнь,Сахарный диабет,cols_for_key,...,PpeakP,PpeakN,Rpeak,Speak,Tpeak,Tons,Toffs,RonsF,RoffsF,SDNN
0,0,0,53.0,1.0,180.0,100.0,0,1.0,1.0,602,...,211.0,156.0,339.0,310.0,584.0,542.0,622.0,32.0,31.0,24.26700
1,0,0,53.0,1.0,180.0,100.0,0,1.0,1.0,602,...,244.0,196.0,371.0,340.0,614.0,570.0,652.0,33.0,31.0,16.32700
2,1,1,68.0,1.0,184.0,130.0,0,1.0,0.0,633,...,0.0,0.0,375.0,441.0,598.0,572.0,654.0,16.0,16.0,2.29900
3,1,1,68.0,1.0,184.0,130.0,0,1.0,0.0,633,...,0.0,0.0,392.0,461.0,636.0,560.0,668.0,16.0,16.0,2.02500
4,0,0,72.0,1.0,175.0,85.0,1,1.0,0.0,610,...,268.0,220.0,391.0,419.0,688.0,660.0,722.0,35.0,34.0,33.84900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4030,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0,...,449.0,400.0,593.0,566.0,940.0,898.0,974.0,34.0,30.0,26.61700
4031,0,0,73.0,2.0,NaN,NaN,0,3.0,0.0,380,...,0.0,442.0,495.0,460.0,816.0,754.0,854.0,31.0,26.0,64.77549
4032,0,0,71.0,1.0,NaN,NaN,0,2.0,0.0,377,...,507.0,606.0,577.0,606.0,914.0,874.0,954.0,34.0,28.0,0.00000
4033,0,0,85.0,NaN,NaN,NaN,0,NaN,NaN,147,...,453.0,400.0,613.0,583.0,972.0,940.0,1010.0,34.0,33.0,120.30420


In [78]:
df_systole = df_systole.replace('нет данных', np.nan)

/tmp/ipykernel_4679/743961919.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_systole = df_systole.replace('нет данных', np.nan)


In [120]:
from sklearn.model_selection import GroupShuffleSplit  
from sklearn.impute import SimpleImputer# <-- Добавляем этот импорт

X = df_systole.loc[:, 'Возраст, годы':'SDNN']
y = df_systole['SystDisf_outcome_AV'].astype(int)
groups = df_systole['cols_for_key']  # Ключ группировки (должен присутствовать в df_systole)

# Заменяем обычный split на групповой, чтобы одинаковые пациенты не попадали в обе выборки
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

custom_weights = {0: 1, 1: 3} 

model_lr = LogisticRegressionCV(penalty = 'elasticnet', solver = 'saga', max_iter = 8000, Cs = [0.1])
model_tr = ExtraTreesClassifier(n_estimators = 800, class_weight = custom_weights, random_state = 42)

ensemble = VotingClassifier(
    estimators=[('lr', model_lr), ('tr', model_tr)],
    voting = 'soft'
)

cv_strategy = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)

Pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  
    ('scaler', RobustScaler()),
    ('clf', ensemble)
])


param_grid = {
    'clf__tr__max_depth': [12],
    'clf__lr__l1_ratios': [[0.1, 0.5, 0.9]],
}
    

grid_search = HalvingGridSearchCV(Pipe, param_grid, cv = cv_strategy, scoring = 'recall', resource = 'n_samples', factor = 3, n_jobs = -1, random_state =42)
grid_search.fit(X_train,y_train)
best_xgb = grid_search.best_estimator_



calibrated_model = CalibratedClassifierCV(best_xgb, method='isotonic', cv=5, n_jobs = -1)
calibrated_model.fit(X_train, y_train)
    

train_probs = calibrated_model.predict_proba(X_train)[:, 1]
fpr, tpr, thresholds = roc_curve(y_train, train_probs)

# --- ИЗМЕНЕНИЕ: Внедрение веса для максимизации Чувствительности (Se) ---
# Настройка: w = 4 означает, что пропустить больного в 4 раза хуже, чем ошибиться на здоровом.
# Вы можете менять это число (например, поставить 3 или 5), чтобы регулировать баланс Se и Sp.
w = 9

# Рассчитываем взвешенный индекс Юдена вместо стандартного (tpr - fpr)
weighted_youden_index = (w * tpr) - fpr
optimal_idx = np.argmax(weighted_youden_index)
youden_threshold = thresholds[optimal_idx]
# ------------------------------------------------------------------------

# 2. Применяем новый смещенный порог на тестовой выборке
test_probs = calibrated_model.predict_proba(X_test)[:, 1]
y_pred_custom = (test_probs >= youden_threshold).astype(int)

/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/an

In [121]:
 # Расчет метрик
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
metrics = {
    'AUC': roc_auc_score(y_test, test_probs),
    'Se (Sens)': tp / (tp + fn),
    'Sp (Spec)': tn / (tn + fp),
    'PPV': tp / (tp + fp),
    'NPV': tn / (tn + fn),
    'Balanced Acc': balanced_accuracy_score(y_test, y_pred_custom),
    'Youden Threshold': youden_threshold
}
    
    



print(metrics)

{'AUC': 0.9075756156943533, 'Se (Sens)': np.float64(0.7288135593220338), 'Sp (Spec)': np.float64(0.888121546961326), 'PPV': np.float64(0.3467741935483871), 'NPV': np.float64(0.9757207890743551), 'Balanced Acc': 0.8084675531416798, 'Youden Threshold': np.float64(0.13484772927970684)}


In [282]:
y.value_counts()

SystDisf_outcome_AV
0    3754
1     276
Name: count, dtype: int64

In [283]:
train_probs = calibrated_model.predict_proba(X_train)[:, 1]
y_train_custom = (train_probs >= youden_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_train, y_train_custom).ravel()
metrics = {
    'AUC': roc_auc_score(y_train, train_probs),
    'Se (Sens)': tp / (tp + fn),
    'Sp (Spec)': tn / (tn + fp),
    'PPV': tp / (tp + fp),
    'NPV': tn / (tn + fn),
    'Balanced Acc': balanced_accuracy_score(y_train, y_train_custom),
    'Youden Threshold': youden_threshold
}
    
    



print(metrics)

{'AUC': 0.9969164954013439, 'Se (Sens)': np.float64(0.9956709956709957), 'Sp (Spec)': np.float64(0.9484848484848485), 'PPV': np.float64(0.6005221932114883), 'NPV': np.float64(0.9996451383960255), 'Balanced Acc': 0.9720779220779221, 'Youden Threshold': np.float64(0.09407968373449496)}


In [105]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, HalvingGridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, precision_recall_curve, classification_report
from sklearn.model_selection import GroupShuffleSplit, HalvingGridSearchCV, cross_val_predict
# 1. Групповое разбиение (пациенты не пересекаются)
X = df_systole.loc[:, 'Возраст, годы':'SDNN']
y = df_systole['SystDisf_outcome_AW'].astype(int)
groups = df_systole['cols_for_key'] 

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 2. Адекватные веса для 3% позитивов
# Для логрега используем balanced, для деревьев усиленный вес
custom_weights = {0: 1, 1: 3}  

model_lr = LogisticRegressionCV(
    penalty='elasticnet', solver='saga', max_iter=8000, 
    Cs=[0.1], l1_ratios=[0.3], class_weight='balanced', cv=5
)
model_tr = ExtraTreesClassifier(
    n_estimators=800, class_weight=custom_weights, max_depth=12, 
    min_samples_leaf=5, random_state=42
)

ensemble = VotingClassifier(
    estimators=[('lr', model_lr), ('tr', model_tr)],
    voting='soft'
)

# ВАЖНО: Сначала импутация, потом масштабирование
Pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('clf', ensemble)
])

param_grid = {
    'clf__tr__max_depth': [12],
    'clf__lr__l1_ratios': [[0.3]]  # пример расширения сетки
}

# Для групповых данных в CV лучше использовать StratifiedGroupKFold (sklearn >= 1.2)
# Если версия старая, оставляем StratifiedKFold, но понимаем ограничение
from sklearn.model_selection import StratifiedKFold
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

grid_search = HalvingGridSearchCV(
    Pipe, param_grid, cv=cv_strategy, scoring='recall', 
    resource='n_samples', factor=3, n_jobs=-1, random_state=42
)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_

# 3. ОБОБЩЁННЫЙ ПОРОГ (Out-of-Fold предсказания по ROC-кривой и взвешенному Юдену)
# Получаем честные вероятности на обучающей выборке без эффекта запоминания
oof_probs = cross_val_predict(best_pipeline, X_train, y_train, cv=cv_strategy, method='predict_proba')[:, 1]

# Строим классическую медицинскую ROC-кривую
fpr, tpr, thresholds_roc = roc_curve(y_train, oof_probs)

# --- УПРАВЛЕНИЕ ЧУВСТВИТЕЛЬНОСТЬЮ ЧЕРЕЗ ВЕС ЮДЕНА ---
# НАСТРОЙКА (W): во сколько раз пропустить больного хуже, чем ложно обвинить здорового.
# По умолчанию Юден использует w=1. Поставим w=15 или даже w=20. 
# Чем БОЛЬШЕ этот коэффициент, тем ВЫШЕ будет финальная чувствительность (Se).
W = 1.8 

# Рассчитываем взвешенный индекс Юдена
weighted_youden_index = (W * tpr) - fpr
optimal_idx = np.argmax(weighted_youden_index)
optimal_threshold = thresholds_roc[optimal_idx]

print(f"Выбранный коэффициент штрафа W: {W}")
print(f"Оптимальный порог отсечения (Взвешенный Юден): {optimal_threshold:.4f}")

# 4. Финальное предсказание на тесте
test_probs = best_pipeline.predict_proba(X_test)[:, 1]
y_pred_test = (test_probs >= optimal_threshold).astype(int)


/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/an

Выбранный коэффициент штрафа W: 1.8
Оптимальный порог отсечения (Взвешенный Юден): 0.3135


In [111]:
from sklearn.metrics import confusion_matrix, roc_auc_score, balanced_accuracy_score

# Расчет метрик на тестовой выборке
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_test).ravel()

metrics = {
    'ROC-AUC': roc_auc_score(y_test, test_probs),
    'Se (Sens)': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
    'Sp (Spec)': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
    'PPV': tp / (tp + fp) if (tp + fp) > 0 else 0.0,
    'NPV': tn / (tn + fn) if (tn + fn) > 0 else 0.0,
    'Balanced Acc': balanced_accuracy_score(y_test, y_pred_test),
    'Weighted Youden Threshold': optimal_threshold
}

# Вывод в консоль
print("\n📊 Итоговые метрики на тесте:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


📊 Итоговые метрики на тесте:
ROC-AUC: 0.8555
Se (Sens): 0.6818
Sp (Spec): 0.8384
PPV: 0.1087
NPV: 0.9891
Balanced Acc: 0.7601
Weighted Youden Threshold: 0.3135


In [110]:
y.value_counts()

SystDisf_outcome_AW
0    3897
1     133
Name: count, dtype: int64

In [112]:
df_total['Syst_DyastDisf_outcome'] = np.where(df_total['Дисфункция миокрада (систола и/или диастола)'] == 1, 1, 0)

df_total['Syst_DyastDisf_outcome']
rows_mask = df_total[['Syst_DyastDisf_outcome']].isna().all(axis=1)
cols_to_select = ['Syst_DyastDisf_outcome', 'Возраст, годы', 'Пол', 'Рост, см', 'Вес, кг', 'Курение', 'Гипертоническая болезнь', 'Сахарный диабет'] + df_total.loc[:, 'RR':'SDNN'].columns.tolist()

min_filled = int(len(cols_to_select) * 0.8)
df_syst_dyast = df_total.loc[~rows_mask, cols_to_select].dropna(thresh=min_filled)

In [113]:
df_syst_dyast= df_syst_dyast.replace('нет данных', np.nan)

/tmp/ipykernel_4679/3603184249.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_syst_dyast= df_syst_dyast.replace('нет данных', np.nan)


In [126]:
from sklearn.model_selection import GroupShuffleSplit  
from sklearn.impute import SimpleImputer# <-- Добавляем этот импорт

X = df_syst_dyast.loc[:, 'Возраст, годы':'SDNN']  # используем df_syst_dyast
y = df_syst_dyast['Syst_DyastDisf_outcome'].astype(int)
groups = df_systole['cols_for_key']  # cols_for_key тоже из df_syst_dyast

# Заменяем обычный split на групповой, чтобы одинаковые пациенты не попадали в обе выборки
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

custom_weights = {0: 1, 1: 3} 

model_lr = LogisticRegressionCV(penalty = 'elasticnet', solver = 'saga', max_iter = 8000, Cs = [0.1])
model_tr = ExtraTreesClassifier(n_estimators = 1000, class_weight = custom_weights, random_state = 42)

ensemble = VotingClassifier(
    estimators=[('lr', model_lr), ('tr', model_tr)],
    voting = 'soft'
)

cv_strategy = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)

Pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('clf', ensemble)
])


param_grid = {
    'clf__tr__max_depth': [12],
    'clf__lr__l1_ratios': [[0.1, 0.5, 0.9]],
}
    

grid_search = HalvingGridSearchCV(Pipe, param_grid, cv = cv_strategy, scoring = 'recall', resource = 'n_samples', factor = 3, n_jobs = -1, random_state =42)
grid_search.fit(X_train,y_train)
best_xgb = grid_search.best_estimator_



calibrated_model = CalibratedClassifierCV(best_xgb, method='isotonic', cv=5, n_jobs = -1)
calibrated_model.fit(X_train, y_train)
    

train_probs = calibrated_model.predict_proba(X_train)[:, 1]
fpr, tpr, thresholds = roc_curve(y_train, train_probs)

# --- ИЗМЕНЕНИЕ: Внедрение веса для максимизации Чувствительности (Se) ---
# Настройка: w = 4 означает, что пропустить больного в 4 раза хуже, чем ошибиться на здоровом.
# Вы можете менять это число (например, поставить 3 или 5), чтобы регулировать баланс Se и Sp.
w = 3

# Рассчитываем взвешенный индекс Юдена вместо стандартного (tpr - fpr)
weighted_youden_index = (w * tpr) - fpr
optimal_idx = np.argmax(weighted_youden_index)
youden_threshold = thresholds[optimal_idx]
# ------------------------------------------------------------------------

# 2. Применяем новый смещенный порог на тестовой выборке
test_probs = calibrated_model.predict_proba(X_test)[:, 1]
y_pred_custom = (test_probs >= youden_threshold).astype(int)

/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/anaconda3/envs/work/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/denzel/an

In [127]:
 # Расчет метрик
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_custom).ravel()
metrics = {
    'AUC': roc_auc_score(y_test, test_probs),
    'Se (Sens)': tp / (tp + fn),
    'Sp (Spec)': tn / (tn + fp),
    'PPV': tp / (tp + fp),
    'NPV': tn / (tn + fn),
    'Balanced Acc': balanced_accuracy_score(y_test, y_pred_custom),
    'Youden Threshold': youden_threshold
}
    
    



print(metrics)

{'AUC': 0.8161845291194088, 'Se (Sens)': np.float64(0.7203389830508474), 'Sp (Spec)': np.float64(0.7533834586466165), 'PPV': np.float64(0.3413654618473896), 'NPV': np.float64(0.9382022471910112), 'Balanced Acc': 0.736861220848732, 'Youden Threshold': np.float64(0.14222423328305683)}


In [308]:
y.value_counts()

Syst_DyastDisf_outcome
0    3471
1     559
Name: count, dtype: int64